# 2D ZX-diagrams of holographic codes

Minimal notebook — uses `pyzx.zx.draw(...)` directly on graphs built by
`zxholo`.

Covered:
1. {5,4} Pentagon HaPPY code — small layers
2. {4,5} ZX-holographic code — paper Table-1 sizes
3. |+⟩-gauge fixed versions
4. Pauli-web stabiliser overlays
5. Other hyperbolic tilings ({6,4}, {7,3}, {3,7})

Requires `venv_testing` (pyzx Pauli-webs fork + LEGO_HQEC).

**Run inside Jupyter** — the fork's `zx.draw` uses the D3 (JavaScript)
backend by default in notebooks, which accepts both `scale=` and
`pauli_web=` kwargs. A plain Python shell falls back to the
matplotlib backend, which only accepts `figsize=` and ignores
`pauli_web=`.

In [1]:
import sys
from pathlib import Path
# repo layout: notebooks/ sits next to the package.
sys.path.insert(0, str(Path.cwd().parent))

import zxholo as az
import pyzx as zx
from pyzx.webs import compute_stabilisers

# pyzx.draw renders inline; matplotlib is the backend.
%matplotlib inline

## 1. {5,4} Pentagon HaPPY code

The original holographic code. `layers=2` gives the central
pentagon only (a [[5,1,3]] perfect tensor), `layers=3` adds one ring.

In [2]:
g, _ = az.build_zx_holo_generic(p=5, q=4, layers=2)
print(f'{{5,4}} layers=2: {len(list(g.inputs()))} bulk, {len(list(g.outputs()))} boundary')
zx.draw(g, labels=True, scale=20)

[hypertiling] Warning: This kernel is deprecated! Better use the 'GRC' kernel instead!
{5,4} layers=2: 1 bulk, 5 boundary


In [3]:
g, _ = az.build_zx_holo_generic(p=5, q=4, layers=3)
print(f'{{5,4}} layers=3: {len(list(g.inputs()))} bulk, {len(list(g.outputs()))} boundary')
zx.draw(g, labels=False, scale=15)

[hypertiling] Warning: This kernel is deprecated! Better use the 'GRC' kernel instead!
{5,4} layers=3: 6 bulk, 20 boundary


## 2. {4,5} ZX-holographic code (paper Table 1)

`build_tiled_codes(4, 5, n)` uses the paper-canonical layer policy.
Paper's Table-1 layer `n_paper = n - 3`:

| call                       | paper `n` | `N_boundary` |
|----------------------------|-----------|--------------|
| `build_tiled_codes(4,5,3)` | 0         | 4            |
| `build_tiled_codes(4,5,4)` | 1         | 20           |
| `build_tiled_codes(4,5,5)` | 2         | 76           |
| `build_tiled_codes(4,5,6)` | 3         | 284          |

In [4]:
g, _ = az.build_tiled_codes(p=4, q=5, n=3)  # paper n=0
print(f'{{4,5}} n=3 (paper n=0): {len(list(g.inputs()))} bulk, {len(list(g.outputs()))} boundary')
zx.draw(g, labels=True, scale=25)

{4,5} n=3 (paper n=0): 1 bulk, 4 boundary


In [5]:
g, _ = az.build_tiled_codes(p=4, q=5, n=4)  # paper n=1
print(f'{{4,5}} n=4 (paper n=1): {len(list(g.inputs()))} bulk, {len(list(g.outputs()))} boundary')
zx.draw(g, labels=False, scale=18)

{4,5} n=4 (paper n=1): 13 bulk, 20 boundary


In [6]:
g, _ = az.build_tiled_codes(p=4, q=5, n=5)  # paper n=2
print(f'{{4,5}} n=5 (paper n=2): {len(list(g.inputs()))} bulk, {len(list(g.outputs()))} boundary')
zx.draw(g, labels=False, scale=10)

{4,5} n=5 (paper n=2): 61 bulk, 76 boundary


## 3. |+⟩-gauge fixed

Project all bulk legs except `keep_bulk_idx=0` onto |+⟩. This is what
the paper's fig 10/13 use for decoder benchmarks.

In [7]:
g, _ = az.build_tiled_codes(p=4, q=5, n=4)
print(f'before gauge: {len(list(g.inputs()))} bulk inputs')
g_gauged = az.apply_gauge(g, gauge=az.GAUGE_PLUS, keep_bulk_idx=0)
print(f'after  gauge: {len(list(g_gauged.inputs()))} bulk inputs')
zx.draw(g_gauged, labels=False, scale=18)

before gauge: 13 bulk inputs
after  gauge: 1 bulk inputs


## 4. Pauli-web stabiliser overlays

`pyzx.webs.compute_stabilisers` returns every Pauli web of the diagram;
`zx.draw(g, pauli_web=w)` colours edges by each web's Pauli pattern
(X = red, Z = green, Y = blue). We show the first 3 to keep the notebook small.

In [8]:
g, _ = az.build_tiled_codes(p=4, q=5, n=4)
g_gauged = az.apply_gauge(g, gauge=az.GAUGE_PLUS, keep_bulk_idx=0)
webs = compute_stabilisers(g_gauged)
print(f'found {len(webs)} Pauli webs')
for i, web in enumerate(webs[:3]):
    print(f'\nweb {i}:')
    zx.draw(g_gauged, pauli_web=web, scale=18, labels=False)

found 21 Pauli webs

web 0:



web 1:



web 2:


## 5. Other hyperbolic tilings

Any Schläfli symbol with `1/p + 1/q < 1/2` is a valid hyperbolic
tessellation. `build_tiled_codes` (paper-canonical) handles all of
them cleanly; the legacy `build_zx_holo_generic` only gives sensible
output for {5,4} / {4,5} and degrades at larger `n` on other tilings.

**Bulk / boundary counts at `n=4`:**

| tile   | `build_tiled_codes(p,q,4)` | `build_zx_holo_generic(p,q,4)` |
|--------|----------------------------|--------------------------------|
| {5,4}  | 11 bulk / 25 bdry          | 6 / 20                          |
| {4,5}  | 13 / 20                    | 13 / 20                         |
| {6,4}  | 13 / 42                    | 31 / 114  *(distorted)*         |
| {7,3}  | 8 / 21                     | **29 / 28  (inverted!)**        |
| {3,7}  | 16 / 12                    | 10 / 18                         |
| {5,5}  | 16 / 40                    | 26 / 100  *(distorted)*         |

The {7,3} inversion is a bug in the legacy builder's final-layer
cleanup: it only handles outermost cells of degree 1 or 2 (fine for
{5,4} pentagons) and silently fails on higher-degree cases. Use
`build_tiled_codes` below for anything non-pentagon.

In [9]:
for (p, q, n) in [(6, 4, 4), (7, 3, 4), (3, 7, 4), (5, 5, 4)]:
    g, _ = az.build_tiled_codes(p=p, q=q, n=n)
    ni, no = len(list(g.inputs())), len(list(g.outputs()))
    print(f'{{{p},{q}}} build_tiled_codes n={n}: {ni} bulk, {no} boundary')
    zx.draw(g, labels=False, scale=15)

{6,4} build_tiled_codes n=4: 13 bulk, 42 boundary


{7,3} build_tiled_codes n=4: 8 bulk, 21 boundary


{3,7} build_tiled_codes n=4: 16 bulk, 12 boundary


{5,5} build_tiled_codes n=4: 16 bulk, 40 boundary


### For comparison — legacy generic builder at n=4

Included to **show** the distortion, not for use. Compare these to the
cell above.

In [10]:
for (p, q, L) in [(6, 4, 4), (7, 3, 4), (3, 7, 4)]:
    g, _ = az.build_zx_holo_generic(p=p, q=q, layers=L)
    ni, no = len(list(g.inputs())), len(list(g.outputs()))
    print(f'{{{p},{q}}} build_zx_holo_generic layers={L}: {ni} bulk, {no} boundary')
    zx.draw(g, labels=False, scale=12)

[hypertiling] Warning: This kernel is deprecated! Better use the 'GRC' kernel instead!
{6,4} build_zx_holo_generic layers=4: 31 bulk, 114 boundary


[hypertiling] Warning: This kernel is deprecated! Better use the 'GRC' kernel instead!
{7,3} build_zx_holo_generic layers=4: 29 bulk, 28 boundary


[hypertiling] Warning: This kernel is deprecated! Better use the 'GRC' kernel instead!
{3,7} build_zx_holo_generic layers=4: 10 bulk, 18 boundary


## 6. Pentagon code Pauli webs (for completeness)

Same as §4 but on the {5,4} pentagon HaPPY side. Only works at small
layers because the number of webs grows fast.

In [11]:
g, _ = az.build_zx_holo_generic(p=5, q=4, layers=2)
g_gauged = az.apply_gauge(g, gauge=az.GAUGE_PLUS, keep_bulk_idx=0)
webs = compute_stabilisers(g_gauged)
print(f'{{5,4}} pentagon n=0: {len(webs)} webs')
for i, web in enumerate(webs[:2]):
    print(f'\nweb {i}:')
    zx.draw(g_gauged, pauli_web=web, scale=25, labels=True)

[hypertiling] Warning: This kernel is deprecated! Better use the 'GRC' kernel instead!
{5,4} pentagon n=0: 6 webs

web 0:



web 1:
